# Comparación estadística entre las arquitecturas de control (experimento E3)

En esta libreta se comparan la Arquitectura I (un controlador PID con factor de escalamiento ρ, denominado PID proporcional o PID simple en las libretas) y la Arquitectura II (controladores PID duales) a partir del RMSE obtenido en los 18 pacientes del experimento E3 de `PID_Filtro_EMA.ipynb`.


**Procedimiento.**
1. Prueba de normalidad de Shapiro–Wilk sobre el RMSE de cada arquitectura y sobre las diferencias apareadas.
2. Prueba de rangos con signo de Wilcoxon (bilateral).
3. Prueba de equivalencia mediante dos pruebas unilaterales (TOST) con margen δ.
4. Medidas complementarias: estimador de Hodges–Lehmann, correlación rango-biserial e intervalo de confianza *bootstrap*.
5. Efecto del sexo sobre el RMSE (prueba U de Mann–Whitney).
6. Número de parámetros sintonizados que quedaron en los límites del intervalo de búsqueda.


## Etapa 1: Configuración y carga de datos

In [1]:
# =============================================================================
# CONFIGURACIÓN
# =============================================================================
import numpy as np
import pandas as pd
from scipy import stats

RUTA_E3   = "RMSE_Arquitecturas_E3.csv"   # RMSE y parámetros por paciente
ALFA      = 0.05                          # Nivel de significancia
DELTA     = 0.5                           # Margen de equivalencia [unidades de BIS]
DELTAS    = [0.2, 0.25, 0.3, 0.4, 0.5, 1.0]   # Márgenes para el análisis de sensibilidad
N_BOOT    = 20000                         # Remuestreos bootstrap
SEMILLA   = 0                             # Semilla del generador aleatorio

df = pd.read_csv(RUTA_E3)
rmse_I  = df["RMSE_ArqI"].to_numpy()
rmse_II = df["RMSE_ArqII"].to_numpy()
dif     = rmse_I - rmse_II                # Diferencias apareadas
n       = len(dif)

print(f"[INFO] Pacientes: {n}")
print(f"       RMSE Arquitectura I  : {rmse_I.mean():.4f} ± {rmse_I.std(ddof=1):.4f}")
print(f"       RMSE Arquitectura II : {rmse_II.mean():.4f} ± {rmse_II.std(ddof=1):.4f}")
print(f"       Diferencia media     : {dif.mean():.4f} ± {dif.std(ddof=1):.4f}")

[INFO] Pacientes: 18
       RMSE Arquitectura I  : 5.3825 ± 1.5745
       RMSE Arquitectura II : 5.3772 ± 1.6461
       Diferencia media     : 0.0053 ± 0.7347


## Etapa 2: Normalidad y prueba de Wilcoxon

In [2]:
# =============================================================================
# PRUEBA DE SHAPIRO-WILK Y PRUEBA DE RANGOS CON SIGNO DE WILCOXON
# =============================================================================
# La normalidad de las diferencias apareadas es el supuesto que condiciona la
# validez de la prueba t apareada; si se rechaza, se emplea la prueba de Wilcoxon.
for nombre, muestra in [("RMSE Arquitectura I", rmse_I),
                        ("RMSE Arquitectura II", rmse_II),
                        ("Diferencias apareadas", dif)]:
    w, p = stats.shapiro(muestra)
    print(f"Shapiro-Wilk | {nombre:24s}: W = {w:.4f}   valor-p = {p:.4f}   "
          f"{'se rechaza normalidad' if p < ALFA else 'no se rechaza normalidad'}")

prueba = stats.wilcoxon(rmse_I, rmse_II)
print(f"\nWilcoxon (bilateral): estadístico = {prueba.statistic:.1f}   valor-p = {prueba.pvalue:.4f}")
print("Conclusión: " + ("se rechaza la igualdad" if prueba.pvalue < ALFA
                        else "no se rechaza la igualdad (esto, por sí solo, no demuestra equivalencia)"))

Shapiro-Wilk | RMSE Arquitectura I     : W = 0.8673   valor-p = 0.0161   se rechaza normalidad
Shapiro-Wilk | RMSE Arquitectura II    : W = 0.8329   valor-p = 0.0046   se rechaza normalidad
Shapiro-Wilk | Diferencias apareadas   : W = 0.8847   valor-p = 0.0314   se rechaza normalidad

Wilcoxon (bilateral): estadístico = 70.0   valor-p = 0.5226
Conclusión: no se rechaza la igualdad (esto, por sí solo, no demuestra equivalencia)


## Etapa 3: Prueba de equivalencia (TOST)

La prueba de Wilcoxon únicamente permite concluir si existe evidencia de una diferencia; un resultado no significativo no demuestra que las arquitecturas sean equivalentes. La prueba de dos pruebas unilaterales (*Two One-Sided Tests*, TOST) contrasta la hipótesis nula de que la diferencia es de al menos δ en valor absoluto, frente a la alternativa de que se encuentra dentro del intervalo (−δ, δ). Se concluye la equivalencia cuando ambas pruebas unilaterales resultan significativas, es decir, cuando el mayor de los dos valores-p es menor que el nivel de significancia.

In [3]:
# =============================================================================
# PRUEBA DE EQUIVALENCIA TOST BASADA EN LA PRUEBA DE WILCOXON
# =============================================================================
def tost_wilcoxon(diferencias, delta):
    """
    Aplica la prueba de equivalencia TOST a diferencias apareadas mediante dos
    pruebas unilaterales de rangos con signo de Wilcoxon.

    Retorna el valor-p de cada prueba unilateral y el valor-p global (el mayor).
    """
    p_inferior = stats.wilcoxon(diferencias + delta, alternative="greater").pvalue
    p_superior = stats.wilcoxon(diferencias - delta, alternative="less").pvalue
    return p_inferior, p_superior, max(p_inferior, p_superior)


p_inf, p_sup, p_tost = tost_wilcoxon(dif, DELTA)
print(f"TOST con delta = {DELTA}")
print(f"  Prueba unilateral inferior (dif > -delta): valor-p = {p_inf:.4f}")
print(f"  Prueba unilateral superior (dif < +delta): valor-p = {p_sup:.4f}")
print(f"  Valor-p global                           : {p_tost:.4f}")
print("  Conclusión: " + ("se concluye equivalencia" if p_tost < ALFA else "no se concluye equivalencia"))

# Análisis de sensibilidad respecto del margen de equivalencia
sensibilidad = pd.DataFrame(
    [dict(delta=d, valor_p=tost_wilcoxon(dif, d)[2]) for d in DELTAS])
sensibilidad["Equivalencia"] = sensibilidad["valor_p"] < ALFA
sensibilidad

TOST con delta = 0.5
  Prueba unilateral inferior (dif > -delta): valor-p = 0.0028
  Prueba unilateral superior (dif < +delta): valor-p = 0.0028
  Valor-p global                           : 0.0028
  Conclusión: se concluye equivalencia


,delta,valor_p,Equivalencia
0,0.20,0.196369,False
1,0.25,0.114376,False
2,0.30,0.070759,False
3,0.40,0.015182,True
4,0.50,0.002800,True
5,1.00,0.000095,True


## Etapa 4: Medidas complementarias

In [4]:
# =============================================================================
# HODGES-LEHMANN, CORRELACIÓN RANGO-BISERIAL E INTERVALO BOOTSTRAP
# =============================================================================
# Estimador de Hodges-Lehmann: mediana de los promedios de Walsh
walsh = [(dif[i] + dif[j]) / 2 for i in range(n) for j in range(i, n)]
hl = float(np.median(walsh))

# Correlación rango-biserial apareada: (R+ − R−) / (R+ + R−)
no_nulas = dif[dif != 0]
rangos   = stats.rankdata(np.abs(no_nulas))
r_mas, r_menos = rangos[no_nulas > 0].sum(), rangos[no_nulas < 0].sum()
r_rb = (r_mas - r_menos) / (r_mas + r_menos)

# Intervalo de confianza al 90 % de la diferencia media mediante bootstrap
generador = np.random.default_rng(SEMILLA)
medias = [generador.choice(dif, n).mean() for _ in range(N_BOOT)]
ic_inf, ic_sup = np.percentile(medias, [5, 95])

print(f"Estimador de Hodges-Lehmann            : {hl:.3f}")
print(f"Correlación rango-biserial (r_rb)      : {r_rb:.2f}")
print(f"IC al 90 % de la diferencia media      : [{ic_inf:.2f}, {ic_sup:.2f}]")
print(f"Pacientes en que la Arquitectura I es mejor: {(dif < 0).sum()}/{n}")

Estimador de Hodges-Lehmann            : 0.110
Correlación rango-biserial (r_rb)      : 0.18
IC al 90 % de la diferencia media      : [-0.29, 0.26]
Pacientes en que la Arquitectura I es mejor: 7/18


## Etapa 5: Efecto del sexo y parámetros en los límites del intervalo de búsqueda

In [5]:
# =============================================================================
# EFECTO DEL SEXO SOBRE EL RMSE
# =============================================================================
for arq, col in [("I", "RMSE_ArqI"), ("II", "RMSE_ArqII")]:
    h = df.loc[df["Sexo"] == "M", col]
    m = df.loc[df["Sexo"] == "F", col]
    u = stats.mannwhitneyu(h, m)
    print(f"Arquitectura {arq:2s}: hombres {h.mean():.2f}   mujeres {m.mean():.2f}   "
          f"U de Mann-Whitney valor-p = {u.pvalue:.2e}")

# =============================================================================
# PARÁMETROS EN LOS LÍMITES DEL INTERVALO DE BÚSQUEDA
# =============================================================================
LIM_INF = 1e-7                                   # Tolerancia para el límite inferior (1e-8)
limites_sup = {"ArqI_rho": 1.0}                  # Límite superior de rho en E3
columnas = [c for c in df.columns if c.startswith("Arq") and not c.startswith("RMSE")]
resumen_limites = pd.DataFrame([
    dict(Parametro=c,
         En_limite_inferior=int((df[c] <= LIM_INF).sum()),
         En_limite_superior=int((df[c] >= limites_sup.get(c, 2.5) - 1e-6).sum()))
    for c in columnas])
resumen_limites

Arquitectura I : hombres 7.02   mujeres 4.07   U de Mann-Whitney valor-p = 4.57e-05
Arquitectura II: hombres 7.08   mujeres 4.02   U de Mann-Whitney valor-p = 4.57e-05


,Parametro,En_limite_inferior,En_limite_superior
0,ArqI_Kp,0,0
1,ArqI_Ki,9,0
2,ArqI_Kd,0,1
3,ArqI_rho,6,2
4,ArqII_Kp_p,2,0
5,ArqII_Ki_p,12,0
6,ArqII_Kd_p,2,0
7,ArqII_Kp_r,2,1
8,ArqII_Ki_r,9,0
9,ArqII_Kd_r,0,3
